In [2]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
# Load Data
data = pd.read_csv('/home/susan/mof-co2-adsorption/data/processed/data_clean_v2')
# Copy dataframe
df= data.copy()
df.columns

Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg'],
      dtype='object')

In [10]:
# Store the four structural features used in the relationship analysis.
feature_columns = ["lcd", "pld", "void_fraction", "surface_area_m2g"]

# Store the five CO2-uptake targets in increasing pressure order.
target_columns = ["CO2_uptake_0.01bar_molkg", "CO2_uptake_0.05bar_molkg", "CO2_uptake_0.1bar_molkg", "CO2_uptake_0.5bar_molkg", "CO2_uptake_2.5bar_molkg"]


In [ ]:
# Examine the five CO₂ target distributions.
# Step 1A — Create a numerical summary:
        # Start by measuring each:
        # target’s range, median, skewness,
        #  and number of zero-uptake values.

# Calculate descriptive statistics for each CO2-uptake target.
target_summary = df[target_columns].describe().T # The .T changes the table orientation:With .T: 
#each pressure appears as a row, making pressure comparison easier.

# Display the descriptive-statistics table.
display(target_summary.round(3))

print('='*30)

# Calculate the skewness of each target distribution.
target_summary["skewness"] = df[target_columns].skew()

# Count the number of zero-uptake records at each pressure.
target_summary["zero_count"] = df[target_columns].eq(0).sum()

# Calculate the percentage of zero-uptake records at each pressure.
target_summary["zero_percent"] = df[target_columns].eq(0).mean()*100

# Select the most useful statistics for interpreting the distributions.
target_summary = target_summary[["count", "mean", "std", "min", "25%", "50%", "75%", "max", "skewness", "zero_count", "zero_percent"]]

# Display the target-distribution summary rounded to three decimal places.
display(target_summary.round(3))


,count,mean,std,min,25%,50%,75%,max
CO2_uptake_0.01bar_molkg,31234.0,0.127,0.223,0.0,0.022,0.047,0.124,3.622
CO2_uptake_0.05bar_molkg,31234.0,0.445,0.534,0.0,0.111,0.232,0.561,5.475
CO2_uptake_0.1bar_molkg,31234.0,0.740,0.748,0.0,0.221,0.452,1.007,6.459
CO2_uptake_0.5bar_molkg,31234.0,2.196,1.458,0.0,1.033,1.825,3.058,9.157
CO2_uptake_2.5bar_molkg,31234.0,5.685,2.554,0.0,3.687,5.415,7.402,15.753


,count,mean,std,min,25%,50%,75%,max,skewness,zero_count,zero_percent
CO2_uptake_0.01bar_molkg,31234.0,0.127,0.223,0.0,0.022,0.047,0.124,3.622,4.537,53,0.170
CO2_uptake_0.05bar_molkg,31234.0,0.445,0.534,0.0,0.111,0.232,0.561,5.475,2.503,25,0.080
CO2_uptake_0.1bar_molkg,31234.0,0.740,0.748,0.0,0.221,0.452,1.007,6.459,1.915,15,0.048
CO2_uptake_0.5bar_molkg,31234.0,2.196,1.458,0.0,1.033,1.825,3.058,9.157,1.000,10,0.032
CO2_uptake_2.5bar_molkg,31234.0,5.685,2.554,0.0,3.687,5.415,7.402,15.753,0.472,7,0.022


| Statistic      | Meaning                                            |
| -------------- | -------------------------------------------------- |
| `count`        | Number of MOFs included                            |
| `mean`         | Average CO₂ uptake                                 |
| `std`          | How much uptake varies among MOFs                  |
| `min`          | Lowest uptake                                      |
| `25%`          | 25% of MOFs have uptake below this value           |
| `50%`          | Median or middle uptake                            |
| `75%`          | 75% of MOFs have uptake below this value           |
| `max`          | Highest uptake                                     |
| `skewness`     | Indicates whether the distribution has a long tail |
| `zero_count`   | Number of records with zero uptake                 |
| `zero_percent` | Percentage of records with zero uptake             |



## Interpretation of CO₂ Target Distributions

All five targets contain 31,234 observations, confirming that there are no missing CO₂-uptake values.

### Results by Pressure
| Pressure | Median uptake (mol/kg) | Skewness | Zero uptake | Interpretation                                                                          |
| -------- | ---------------------: | -------: | ----------: | --------------------------------------------------------------------------------------- |
| 0.01 bar |                  0.047 |    4.537 | 53 (0.170%) | Most MOFs have very low uptake, but a small number have exceptionally high uptake       |
| 0.05 bar |                  0.232 |    2.503 | 25 (0.080%) | Uptake increases, but the distribution remains strongly right-skewed                    |
| 0.1 bar  |                  0.452 |    1.915 | 15 (0.048%) | Most MOFs adsorb more CO₂, although high-performing MOFs still create a long upper tail |
| 0.5 bar  |                  1.825 |    1.000 | 10 (0.032%) | Uptake is substantially higher and the distribution is less strongly skewed             |
| 2.5 bar  |                  5.415 |    0.472 |  7 (0.022%) | Uptake is highest and more evenly distributed across the MOFs                           |

### 1. Uptake Increases with Pressure

* Both the mean and median CO₂ uptake increase continuously across the five pressure levels:

  * **Mean:** 0.127 → 5.685 mol/kg
  * **Median:** 0.047 → 5.415 mol/kg
* These results show that the overall CO₂ uptake is higher at higher pressures.
* However, this summary does not prove the mechanism causing the increase or confirm that uptake increases for every individual MOF.

### 2. Low-Pressure Uptake Is Strongly Right-Skewed

> **Note:** A right-skewed distribution contains many observations with lower values and a smaller number with much higher values, creating a long tail on the right side. Skewness describes the shape of the distribution; it does not explain the cause of that shape.

* At every pressure, the mean is greater than the median, indicating a right-skewed distribution.
* However, skewness decreases substantially as pressure increases:

  * **0.01 bar:** 4.537
  * **0.05 bar:** 2.503
  * **0.1 bar:** 1.915
  * **0.5 bar:** 1.000
  * **2.5 bar:** 0.472
* At **0.01 bar**, most MOFs have low uptake, while a small number have much higher uptake.
* This pattern is consistent with a small group of MOFs having favorable low-pressure adsorption behavior.
* However, the target distribution alone cannot determine whether this pattern is related to pore structure, chemical interactions, or a combination of both.
* At **2.5 bar**, the mean and median are much closer, and the skewness decreases to **0.472**.
* Therefore, the high-pressure uptake distribution is more balanced than the low-pressure distributions, although it remains slightly right-skewed.

### 3. Target Spread Increases with Pressure

* The standard deviation increases from **0.223 mol/kg at 0.01 bar** to **2.554 mol/kg at 2.5 bar**.
* This shows that CO₂-uptake values are more widely distributed in absolute units at higher pressures.
* This difference is important for modeling because error measurements such as **MAE** and **RMSE** may naturally be larger for the higher-pressure targets due to their larger numerical scale.
* Standard deviation describes only the spread of the target values. It does not explain which MOF properties cause the variation.

> **Note:** A larger standard deviation at high pressure does not necessarily mean that the data is less consistent, because the mean uptake also increases. A proportional comparison would require additional analysis using the coefficient of variation.

### 4. Zero-Uptake Values Are Rare

* The number of zero-uptake records decreases from **53 at 0.01 bar** to only **7 at 2.5 bar**.
* Even at the lowest pressure, zero-uptake values represent only **0.170%** of the dataset.
* Therefore:

  * The target variables are **not zero-inflated**.
  * Zero-uptake records should not be removed solely because their value is zero.
  * Specialized methods for handling excess zeros are not currently necessary.

> **Note:** Zero uptake means that the reported CO₂ adsorption value is `0 mol/kg`. It does not mean zero pressure, missing data, or zero surface area.




# 📊 Final Conclusion: Target Distributions (Modeling-Relevant Summary)

---

## ✅ Data Quality Check

* ✔️ All five CO₂ uptake targets contain **31,234 samples**
* ✔️ **No missing values** in any target column
* ✔️ No imputation required for target variables

---

## 📈 Pressure vs CO₂ Uptake Behavior

* CO₂ uptake **increases consistently with pressure**
* Each pressure level represents a **different output scale**
* The model must learn **separate regression behaviors per pressure**

---

## 📉 Distribution Shape (Skewness Analysis)

* All targets are **right-skewed**
* Skewness decreases significantly with pressure:

| Pressure | Skewness | Interpretation                     |
| -------- | -------- | ---------------------------------- |
| 0.01 bar | 4.537    | Highly imbalanced, strong outliers |
| 2.5 bar  | 0.472    | Near-normal, stable distribution   |

### 🔍 Key Insight

* **Low pressure → highly skewed distributions**
* **High pressure → more balanced and Gaussian-like behavior**

---

## 📊 Mean vs Median Behavior

* At **low pressure**:

  * Large gap between mean and median
  * Distribution dominated by a few high-uptake MOFs

* At **high pressure**:

  * Mean ≈ Median
  * More uniform uptake behavior across MOFs

### 🔍 Interpretation

* Low-pressure models will be:

  * ⚠️ sensitive to extreme outliers
* High-pressure models will be:

  * ✅ more stable and predictable

---

## 📏 Variability (Standard Deviation)

* Standard deviation increases with pressure:

| Pressure | Std Dev (mol/kg) |
| -------- | ---------------- |
| 0.01 bar | 0.223            |
| 2.5 bar  | 2.554            |

### 🔍 Key Insight

* Higher pressure → **larger absolute variation in uptake**
* Therefore:

  * ❗ Raw error values are not comparable across pressures
  * ✅ Scaling or normalization is required for fair evaluation

---

## ⚠️ Zero Uptake Values

* Extremely rare: **< 0.17%**
* These are:

  * ✔️ valid physical observations
  * ❌ not missing data

### 🔍 Impact

* No removal or imputation needed
* May slightly affect **low-pressure model stability**

---

## 🧠 Key Implications for Modeling

* The dataset is **not uniformly distributed across pressures**

### Model Behavior Differences:

* 🔵 **Low-pressure models**

  * Highly skewed targets
  * Outlier-sensitive
  * Harder to learn

* 🟢 **High-pressure models**

  * More normally distributed
  * More stable regression behavior

### ✅ Recommended Strategy:

* Train **separate models per pressure**, OR
* Use a **multi-output model with pressure-aware scaling**

---

## 🚀 Next Step

* Analyze how structural features influence CO₂ uptake:

### Key features to investigate:

* `lcd` (largest cavity diameter)
* `pld` (pore limiting diameter)
* `void_fraction`
* `surface_area_m2g`

### Goal:

* Understand **feature–target relationships** before model training
